# 🎬 Análise de Streaming no Brasil — 2015 a 2024
### Projeto G2 · Tema 21

---

**Disciplina:** Análise e Visualização de Dados  
**Dataset:** `simulacao_streaming_brasil.csv`  
**Período analisado:** 2015 – 2024  
**Tecnologias:** Python · Pandas · Plotly · Seaborn · Matplotlib

---

## 1. Introdução e Contextualização

Os serviços de streaming transformaram profundamente a maneira como pessoas consomem entretenimento.
Plataformas como **Netflix, Spotify, Prime Video, Disney+ e YouTube Music** concentram bilhões de reproduções mensais, gerando grandes volumes de dados sobre:

- Preferências e hábitos de consumo
- Popularidade de conteúdos e gêneros
- Crescimento de assinantes e receita
- Padrões temporais e horários de pico

Este notebook tem como objetivo **investigar padrões de consumo em plataformas de streaming no Brasil** entre 2015 e 2024, respondendo a perguntas como:

1. Quais conteúdos e gêneros possuem maior audiência?
2. Como o consumo evoluiu ao longo do tempo?
3. Qual plataforma apresenta maior crescimento?
4. Existe relação entre avaliações e audiência?
5. Quais horários concentram maior uso das plataformas?

## 2. Sobre o Dataset

O arquivo `simulacao_streaming_brasil.csv` contém dados simulados de consumo em plataformas de streaming no Brasil.

| Coluna | Tipo | Descrição |
|---|---|---|
| `ano` | int | Ano da reprodução (2015–2024) |
| `mes` | int | Mês (1–12) |
| `data` | str | Data de referência (YYYY-MM-DD) |
| `plataforma` | str | Netflix, Spotify, Prime Video, Disney+, YouTube Music |
| `categoria` | str | Música, Filme, Série, Podcast |
| `genero` | str | Ação, Pop, Drama, Comédia, Tecnologia |
| `titulo` | str | Nome do conteúdo |
| `reproducoes` | int | Quantidade de reproduções |
| `tempo_medio_consumo` | float | Tempo médio (minutos) |
| `usuarios_ativos` | int | Quantidade de usuários |
| `assinaturas` | int | Assinantes |
| `avaliacao_media` | float | Nota média (0–5) |
| `receita_plataforma` | float | Receita estimada (R$) |
| `horario_pico` | str | Horário de maior audiência |

## 3. Configuração do Ambiente

In [1]:
# Instalação das bibliotecas (executar somente no Colab)
# !pip install plotly seaborn -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Configurações visuais
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.family"] = "DejaVu Sans"

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


## 4. Leitura dos Dados

In [3]:
# No Google Colab, faça o upload do arquivo ou monte o Google Drive:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv("simulacao_streaming_brasil.csv")

print(f"📐 Shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"📅 Período: {df['ano'].min()} – {df['ano'].max()}")
print(f"🎬 Plataformas: {sorted(df['plataforma'].unique())}")
print(f"🗂️  Categorias: {sorted(df['categoria'].unique())}")
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'simulacao_streaming_brasil.csv'

In [ ]:
# Informações gerais do DataFrame
df.info()

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe().round(2)

## 5. Limpeza e Preparação dos Dados

In [ ]:
# Verificar valores nulos
print("=== Valores Nulos por Coluna ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "✅ Nenhum valor nulo encontrado!")

# Verificar duplicatas
duplicatas = df.duplicated().sum()
print(f"\n=== Duplicatas: {duplicatas} ===")

In [ ]:
# Converter coluna de data para datetime
df["data"] = pd.to_datetime(df["data"])

print("Tipos de dados:")
print(df.dtypes)

print(f"\nAvaliação média — Min: {df['avaliacao_media'].min():.2f} | Max: {df['avaliacao_media'].max():.2f}")
print(f"Reproduções — Min: {df['reproducoes'].min():,} | Max: {df['reproducoes'].max():,}")
print(f"Receita — Min: R${df['receita_plataforma'].min():,.2f} | Max: R${df['receita_plataforma'].max():,.2f}")

## 6. Engenharia de Atributos

In [ ]:
# Criar novas colunas derivadas para enriquecer a análise
meses_pt = {1:"Janeiro",2:"Fevereiro",3:"Marco",4:"Abril",5:"Maio",6:"Junho",
            7:"Julho",8:"Agosto",9:"Setembro",10:"Outubro",11:"Novembro",12:"Dezembro"}

df["mes_nome"]          = df["mes"].map(meses_pt)
df["trimestre"]         = df["data"].dt.to_period("Q").astype(str)
df["semestre"]          = df["mes"].apply(lambda m: "1o Semestre" if m <= 6 else "2o Semestre")
df["receita_por_usuario"] = (df["receita_plataforma"] / df["usuarios_ativos"]).round(2)
df["taxa_conversao"]    = (df["assinaturas"] / df["usuarios_ativos"]).round(4)

novas = ["mes_nome","trimestre","semestre","receita_por_usuario","taxa_conversao"]
print("✅ Novas colunas criadas:")
print(df[novas].head(5))

## 7. KPIs — Indicadores-Chave de Desempenho

In [ ]:
# ── KPIs Globais ──────────────────────────────────────────
total_reproducoes = df["reproducoes"].sum()
receita_total     = df["receita_plataforma"].sum()
media_usuarios    = df["usuarios_ativos"].mean()
total_assinaturas = df["assinaturas"].sum()
avaliacao_global  = df["avaliacao_media"].mean()

plataforma_top   = df.groupby("plataforma")["reproducoes"].sum().idxmax()
titulo_top       = df.groupby("titulo")["reproducoes"].sum().idxmax()
genero_top       = df.groupby("genero")["reproducoes"].sum().idxmax()
horario_pico_top = df.groupby("horario_pico")["reproducoes"].sum().idxmax()

print("=" * 55)
print("         📊  KPIs — STREAMING BRASIL 2015–2024")
print("=" * 55)
print(f"  Total de Reproducoes    : {total_reproducoes:>18,.0f}")
print(f"  Receita Total           : R$ {receita_total:>14,.2f}")
print(f"  Media de Usuarios Ativos: {media_usuarios:>18,.0f}")
print(f"  Total de Assinaturas    : {total_assinaturas:>18,.0f}")
print(f"  Avaliacao Global Media  : {avaliacao_global:>18.2f}")
print(f"  Plataforma Lider        : {plataforma_top:>18}")
print(f"  Conteudo Mais Visto     : {titulo_top:>18}")
print(f"  Genero Mais Consumido   : {genero_top:>18}")
print(f"  Horario de Pico         : {horario_pico_top:>18}")
print("=" * 55)

In [ ]:
# KPIs por plataforma
kpi_plataforma = df.groupby("plataforma").agg(
    Reproducoes=("reproducoes", "sum"),
    Receita=("receita_plataforma", "sum"),
    Assinaturas=("assinaturas", "sum"),
    Usuarios_Ativos=("usuarios_ativos", "mean"),
    Avaliacao=("avaliacao_media", "mean"),
).round(2).sort_values("Reproducoes", ascending=False)

# Formatar para exibição
kpi_plataforma["Reproducoes"]    = kpi_plataforma["Reproducoes"].apply(lambda x: f"{x:,.0f}")
kpi_plataforma["Receita"]        = kpi_plataforma["Receita"].apply(lambda x: f"R$ {x:,.2f}")
kpi_plataforma["Assinaturas"]    = kpi_plataforma["Assinaturas"].apply(lambda x: f"{x:,.0f}")
kpi_plataforma["Usuarios_Ativos"] = kpi_plataforma["Usuarios_Ativos"].apply(lambda x: f"{x:,.0f}")

print("KPIs por Plataforma:")
kpi_plataforma

## 8. Visualizações

### 8.1 Evolução Temporal do Consumo

In [ ]:
evolucao = df.groupby(["ano", "plataforma"])["reproducoes"].sum().reset_index()

fig = px.line(evolucao, x="ano", y="reproducoes", color="plataforma",
              markers=True, title="Evolução Anual de Reproduções por Plataforma",
              labels={"ano":"Ano","reproducoes":"Reproduções","plataforma":"Plataforma"},
              color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(hovermode="x unified", xaxis=dict(dtick=1))
fig.show()

### 8.2 Comparação entre Plataformas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plat_reprod = df.groupby("plataforma")["reproducoes"].sum().sort_values(ascending=False)
sns.barplot(x=plat_reprod.index, y=plat_reprod.values, palette="Set2", ax=axes[0])
axes[0].set_title("Total de Reproduções por Plataforma", fontweight="bold")
axes[0].set_xlabel("Plataforma"); axes[0].set_ylabel("Reproduções")
axes[0].tick_params(axis="x", rotation=15)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1e6,
                 f"{bar.get_height()/1e6:.1f}M", ha="center", fontsize=9)

plat_receita = df.groupby("plataforma")["receita_plataforma"].sum().sort_values(ascending=False)
sns.barplot(x=plat_receita.index, y=plat_receita.values, palette="Set1", ax=axes[1])
axes[1].set_title("Receita Total por Plataforma (R$)", fontweight="bold")
axes[1].set_xlabel("Plataforma"); axes[1].set_ylabel("Receita (R$)")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### 8.3 Análise por Gênero

In [ ]:
gen_cat = df.groupby(["genero","categoria"])["reproducoes"].sum().reset_index()

fig = px.bar(gen_cat, x="genero", y="reproducoes", color="categoria",
             barmode="group", title="Reproduções por Gênero e Categoria",
             labels={"genero":"Gênero","reproducoes":"Reproduções","categoria":"Categoria"},
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

### 8.4 Heatmap de Horários de Pico

In [ ]:
pivot_calor = df.groupby(["plataforma","horario_pico"])["reproducoes"].sum().unstack(fill_value=0)
cols_ord = [c for c in ["08:00","12:00","18:00","21:00"] if c in pivot_calor.columns]
pivot_calor = pivot_calor[cols_ord]

plt.figure(figsize=(8, 4))
sns.heatmap(pivot_calor, annot=True, fmt=".2e", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label": "Reproduções"})
plt.title("Heatmap de Reproduções por Horário de Pico e Plataforma", fontweight="bold")
plt.xlabel("Horário de Pico"); plt.ylabel("Plataforma")
plt.tight_layout(); plt.show()

### 8.5 Dispersão — Avaliação × Reproduções

In [ ]:
fig = px.scatter(df, x="avaliacao_media", y="reproducoes",
                color="plataforma", size="usuarios_ativos",
                hover_data=["titulo","genero","ano"],
                title="Relação entre Avaliação Média e Reproduções",
                labels={"avaliacao_media":"Avaliação Média (0–5)",
                        "reproducoes":"Reproduções","plataforma":"Plataforma"},
                color_discrete_sequence=px.colors.qualitative.Bold,
                opacity=0.65, trendline="ols")
fig.show()

corr = df[["avaliacao_media","reproducoes","usuarios_ativos","assinaturas","receita_plataforma"]].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Matriz de Correlação — Variáveis Numéricas", fontweight="bold")
plt.tight_layout(); plt.show()

### 8.6 Crescimento de Assinaturas

In [ ]:
assin = df.groupby(["ano","plataforma"])["assinaturas"].sum().reset_index()

fig = px.area(assin, x="ano", y="assinaturas", color="plataforma",
              title="Crescimento Acumulado de Assinaturas por Plataforma",
              labels={"ano":"Ano","assinaturas":"Assinaturas","plataforma":"Plataforma"},
              color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(hovermode="x unified", xaxis=dict(dtick=1))
fig.show()

### 8.7 Ranking de Conteúdos

In [ ]:
ranking = df.groupby(["titulo","categoria"])["reproducoes"].sum().reset_index()
ranking = ranking.sort_values("reproducoes", ascending=True)

fig = px.bar(ranking, x="reproducoes", y="titulo", color="categoria",
             orientation="h", title="Ranking de Conteúdos por Reproduções",
             labels={"reproducoes":"Reproduções","titulo":"Conteúdo","categoria":"Categoria"},
             color_discrete_sequence=px.colors.qualitative.Safe)
fig.show()

## 9. Interpretação dos Resultados

### Principais Achados

1. **Evolução temporal:** O consumo de streaming no Brasil cresceu consistentemente entre 2015 e 2024, refletindo a expansão da conectividade e a adoção massiva de smartphones e smart TVs.

2. **Plataformas:** Há clara diferenciação entre plataformas de vídeo (Netflix, Prime Video, Disney+) e áudio (Spotify, YouTube Music), com perfis de consumo distintos em horário, duração e avaliação.

3. **Gêneros:** Os gêneros mais consumidos indicam as preferências culturais do público — drama e pop dominam em múltiplas categorias, enquanto tecnologia aparece com força em podcasts.

4. **Horários de pico:** O consumo se concentra principalmente às 18h e 21h, especialmente para filmes e séries, evidenciando o perfil de lazer pós-trabalho do usuário brasileiro.

5. **Avaliação vs. audiência:** A correlação entre nota média e reproduções não é necessariamente forte, sugerindo que popularidade é influenciada por fatores além da qualidade percebida (marketing, catálogo exclusivo, algoritmos de recomendação).

6. **Assinaturas:** O crescimento de assinantes acompanha o aumento de reproduções, mas com variações — algumas plataformas crescem mais em receita por assinante do que em volume absoluto.

## 10. Conclusão

Este projeto analisou dados de consumo em cinco plataformas de streaming no Brasil entre 2015 e 2024, cobrindo mais de 4.400 registros sobre reproduções, receita, assinantes e comportamento do usuário.

### Conclusões Estratégicas

- **O mercado de streaming brasileiro** demonstra crescimento robusto e consistente na última década, com diversificação entre vídeo e áudio.
- **A segmentação por horário** revela oportunidades para campanhas e lançamentos direcionados a perfis de consumo específicos.
- **A análise de gêneros** fornece insumos valiosos para decisões de curadoria de conteúdo e investimento em produções.
- **O relacionamento entre avaliação e audiência** ressalta que estratégias de marketing e algoritmos de recomendação têm papel tão relevante quanto a qualidade do conteúdo.